## 3. Dataset merging
After cleaning all three main datasets, it is now time to merge all data into the same table to work on our exploratory data analysis.  
But before merging, it is best practice to make a final check of harmonization data.

### Final checks

In [1]:
# import libraries
import pandas as pd
from pathlib import Path
import warnings

# filter User and Filter Warnings for privacy matters
warnings.simplefilter(action="ignore", category=UserWarning)
warnings.simplefilter(action="ignore", category=FutureWarning)

# define clean dataset folder path
clean_dataset_path = Path('../dataset/clean/updated')

# access cleaned datasets
prices_data = pd.read_csv(clean_dataset_path / 'pun-index-clean.csv')
gen_data = pd.read_csv(clean_dataset_path / 'gen-clean.csv')
load_data = pd.read_csv(clean_dataset_path / 'load-italy-clean.csv')

In [2]:
prices_data

,datetime,Date,Prices
0,2023-01-01 00:00:00,2023-01-01,195.9
1,2023-01-01 01:00:00,2023-01-01,191.09
2,2023-01-01 02:00:00,2023-01-01,187.95
3,2023-01-01 03:00:00,2023-01-01,187.82
4,2023-01-01 04:00:00,2023-01-01,187.74
...,...,...,...
30642,2026-06-30 19:00:00,2026-06-30,204.1062
30643,2026-06-30 20:00:00,2026-06-30,198.16
30644,2026-06-30 21:00:00,2026-06-30,182.0325
30645,2026-06-30 22:00:00,2026-06-30,173.3031


In [3]:
gen_data

,Date,Geothermal,Hydro,Photovoltaic,Self-consumption,Thermal,Wind,TOT Generation,Renewables weight
0,2023-01-01 00:00:00,0.62,1.81,0.00,1.443,14.47,0.20,17.10,0.153801
1,2023-01-01 01:00:00,0.62,1.66,0.00,1.365,13.85,0.16,16.29,0.149785
2,2023-01-01 02:00:00,0.62,1.61,0.00,1.260,12.93,0.16,15.32,0.156005
3,2023-01-01 03:00:00,0.62,1.61,0.00,1.191,12.32,0.18,14.73,0.163612
4,2023-01-01 04:00:00,0.62,1.56,0.00,1.081,11.99,0.22,14.39,0.166782
...,...,...,...,...,...,...,...,...,...
30543,2026-06-28 19:00:00,0.56,7.50,1.85,3.359,21.10,0.98,31.99,0.340419
30544,2026-06-28 20:00:00,0.56,8.77,0.19,2.801,21.92,0.73,32.17,0.318620
30545,2026-06-28 21:00:00,0.56,8.43,0.00,2.858,22.61,0.58,32.18,0.297390
30546,2026-06-28 22:00:00,0.56,7.76,0.00,2.790,22.60,0.53,31.45,0.281399


In [4]:
load_data

,Date,Total Load,Load forecasted
0,2023-01-01 00:00:00,21644.24950,22366.75050
1,2023-01-01 01:00:00,20891.00000,20775.74975
2,2023-01-01 02:00:00,19762.24925,19644.24900
3,2023-01-01 03:00:00,18765.50050,18837.25050
4,2023-01-01 04:00:00,18073.75000,18715.25000
...,...,...,...
30595,2026-06-28 19:00:00,43487.25000,42911.50000
30596,2026-06-28 20:00:00,43231.00000,42299.00000
30597,2026-06-28 21:00:00,43050.25100,42659.74975
30598,2026-06-28 22:00:00,41414.99975,41477.75050


It is noticeable that <code>prices_data</code> dataset has *Date* column which has different type of data, and that Date data we want to include is in column called *datetime*.  
In this phase some adjustments are required.

In [5]:
# remove old Date column
prices_data = prices_data.drop(columns='Date')

# rename datetime column into Date
prices_data = prices_data.rename(columns={
    'datetime': 'Date'
})

prices_data

,Date,Prices
0,2023-01-01 00:00:00,195.9
1,2023-01-01 01:00:00,191.09
2,2023-01-01 02:00:00,187.95
3,2023-01-01 03:00:00,187.82
4,2023-01-01 04:00:00,187.74
...,...,...
30642,2026-06-30 19:00:00,204.1062
30643,2026-06-30 20:00:00,198.16
30644,2026-06-30 21:00:00,182.0325
30645,2026-06-30 22:00:00,173.3031


In [6]:
prices_data.describe()

,Date,Prices
count,30647,30647
unique,30644,20089
top,2023-10-29 23:00:00,105.1
freq,2,180


Now that the column uniformation is adjusted, we have to look for duplicated Date.

In [7]:
print(prices_data['Date'].duplicated().sum())
print(gen_data['Date'].duplicated().sum())
print(load_data['Date'].duplicated().sum())

3
0
0


Since <code>prices_data</code> has duplicates, because of Solar Time, correction of this inconsistency is needed.    
The approach applied is the deletion of duplicated rows, which are 5 out of 43824 (0.001%).

In [8]:
# delete duplicated Date rows
prices_data = prices_data.drop_duplicates(
    subset='Date', 
    keep='first'
)

In [9]:
prices_data.describe()

,Date,Prices
count,30644,30644
unique,30644,20087
top,2023-01-01 00:00:00,105.1
freq,1,180


### Progressive merge
Now that tables are normalized, it is time to progressively merge the dataset.

In [10]:
# merge prices_data and gen_data  
merged_data = prices_data.merge(
    gen_data,
    on='Date', 
    how='left'
)

# merge prices_data, gen_data and load_data in merged_data
merged_data = merged_data.merge(
    load_data,
    on='Date', 
    how='left'
)

In [11]:
# convert Date to datetime object again (the merge modified its Type)
merged_data['Date'] = pd.to_datetime(merged_data['Date'])

In [12]:
# sort Date just in case
merged_data = merged_data.sort_values('Date')

In [13]:
# find NaN rows
merged_data.isna().sum().sort_values(ascending=False)


Hydro                100
Geothermal           100
Self-consumption     100
Photovoltaic         100
TOT Generation       100
Renewables weight    100
Thermal              100
Wind                 100
Total Load           100
Load forecasted      100
Date                   0
Prices                 0
dtype: int64

In [14]:
# drop rows with NaN values 
merged_data = merged_data.dropna().reset_index(drop=True)

In [15]:
merged_data['Date'].diff().value_counts().head()

Date
0 days 01:00:00    30534
0 days 02:00:00        8
2 days 01:00:00        1
Name: count, dtype: int64

Now we can export the merged dataset for the next phase, which is proper data analysis.

In [16]:
# export merged_data
merged_data.to_csv(clean_dataset_path / 'merged-dataset.csv', index=False)